### 3.9.31. Active-Set Methods

$$
\min_{\mathbf{x}}\; \tfrac12\mathbf{x}^\top Q \mathbf{x} + \mathbf{c}^\top\mathbf{x}
\quad\text{s.t.}\quad \mathbf{a}_i^\top\mathbf{x} \le b_i,
$$
solved by iterating over a *working set* $\mathcal{W}$ of constraints treated as equalities.

**Explanation:**

Active-set methods solve inequality-constrained [quadratic programs](../04_Convex_Problem_Classes/02_quadratic_programming.ipynb) by guessing which constraints hold with equality at the solution — the *active set* — and solving the resulting equality-constrained subproblem, which reduces to a linear [KKT system](./38_kkt_systems_and_block_elimination.ipynb). If a computed multiplier is negative the corresponding constraint is dropped from the working set; if a step would violate an inactive constraint it is added. The iteration terminates when the working set is correct: the subproblem solution is feasible and all multipliers are nonnegative. This combinatorial search over active sets underlies simplex-like QP solvers.

**Numerical Example:**

$$
\min_{x_1,x_2}\; (x_1 - 2)^2 + (x_2 - 2)^2
\quad\text{s.t.}\quad x_1 + x_2 \le 2,\; x_1\ge 0,\; x_2\ge 0.
$$

The unconstrained minimizer $(2,2)$ violates $x_1+x_2\le 2$, so guess the working set $\mathcal{W}=\{x_1+x_2=2\}$. Solving the equality-constrained problem gives the projection $\mathbf{x}=(1,1)$ with multiplier
$$
\lambda = 2\ \ge 0,
$$
and $(1,1)$ satisfies $x_1,x_2\ge 0$. Both optimality tests pass, so $\mathbf{x}^\star=(1,1)$ with value $2$ is optimal.

In [1]:
import sympy as sp

x_1, x_2, lam = sp.symbols("x_1 x_2 lambda", real=True)
objective = (x_1 - 2)**2 + (x_2 - 2)**2
active_constraint = x_1 + x_2 - 2
lagrangian = objective + lam*active_constraint

stationarity = [sp.diff(lagrangian, variable) for variable in (x_1, x_2)]
solution = sp.solve(stationarity + [active_constraint], [x_1, x_2, lam], dict=True)[0]
minimizer = (solution[x_1], solution[x_2])
multiplier = solution[lam]

print("working set {x_1 + x_2 = 2} solution:", minimizer)
print("multiplier lambda =", multiplier, "(>= 0, so constraint stays active)")
print("bounds x_1, x_2 >= 0 satisfied:", minimizer[0] >= 0 and minimizer[1] >= 0)
print("minimum value =", objective.subs(solution))

working set {x_1 + x_2 = 2} solution: (1, 1)
multiplier lambda = 2 (>= 0, so constraint stays active)
bounds x_1, x_2 >= 0 satisfied: True
minimum value = 2


**Equivalent casadi implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
objective = (decision_variable[0] - 2)**2 + (decision_variable[1] - 2)**2
constraint = decision_variable[0] + decision_variable[1]

solver = ca.qpsol("solver", "qpoases", {"x": decision_variable, "f": objective, "g": constraint},
                  {"printLevel": "none"})
solution = solver(lbg=-ca.inf, ubg=2, lbx=0, ubx=ca.inf)

print("QP minimizer =", solution["x"])
print("minimum value =", float(solution["f"]))


qpOASES -- An Implementation of the Online Active Set Strategy.
Copyright (C) 2007-2015 by Hans Joachim Ferreau, Andreas Potschka,
Christian Kirches et al. All rights reserved.

qpOASES is distributed under the terms of the 
GNU Lesser General Public License 2.1 in the hope that it will be 
useful, but WITHOUT ANY WARRANTY; without even the implied warranty 
of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. 
See the GNU Lesser General Public License for more details.


qpOASES -- An Implementation of the Online Active Set Strategy.
Copyright (C) 2007-2015 by Hans Joachim Ferreau, Andreas Potschka,
Christian Kirches et al. All rights reserved.

qpOASES is distributed under the terms of the 
GNU Lesser General Public License 2.1 in the hope that it will be 
useful, but WITHOUT ANY WARRANTY; without even the implied warranty 
of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. 
See the GNU Lesser General Public License for more details.

QP minimizer = [1, 1]
minimum value = 2.

**References:**

[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.](https://doi.org/10.1007/978-0-387-40065-5)  
[📘 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.](https://web.stanford.edu/~boyd/cvxbook/)

---

[⬅️ Previous: Finite Difference Derivative Approximations](./30_finite_difference_derivative_approximations.ipynb) | [Next: Equality Constrained Convex Quadratic Problems ➡️](./32_equality_constrained_convex_quadratic_problems.ipynb)